In [20]:
with open("the-verdict.txt", "r",encoding="utf-8") as f:
  text = f.read()

print(len(text))
print(text[:100])

20480
I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no g


In [ ]:
# !pip3 install tiktoken
import tiktoken
tokenizer = tiktoken.get_encoding("gpt2")

In [22]:
enc_text = tokenizer.encode(text)
print(len(enc_text))

5146


In [23]:
enc_sample = enc_text[50:]

In [24]:
context_length = 4
x = enc_sample[:context_length]
y = enc_sample[1:context_length+1]
print(f"x: {x}")
print(f"y:    {y}")

x: [290, 4920, 2241, 287]
y:    [4920, 2241, 287, 257]


In [25]:
for i in range(1,context_length+1):
  context = enc_sample[:i]
  desired = enc_sample[i]

  print(context, "------>",desired)

[290] ------> 4920
[290, 4920] ------> 2241
[290, 4920, 2241] ------> 287
[290, 4920, 2241, 287] ------> 257


In [26]:
for i in range(1,context_length+1):
  context = enc_sample[:i]
  desired = enc_sample[i]
  print(tokenizer.decode(context), "------>",tokenizer.decode([desired]))

 and ------>  established
 and established ------>  himself
 and established himself ------>  in
 and established himself in ------>  a


In [27]:
import torch
from torch.utils.data import Dataset, DataLoader

In [28]:

class Gpt2Dataset(Dataset):
  def __init__(self, text, tokenizer, context_length, stride):
    self.input_ids = []
    self.target_ids = []
    token_ids = tokenizer.encode(text)

    for i in range(0, len(token_ids) - context_length, stride):
      context = token_ids[i:i+context_length]
      desired = token_ids[i+1:i+1+context_length]
      self.input_ids.append(torch.tensor(context))
      self.target_ids.append(torch.tensor(desired))

  def __len__(self):
    return len(self.input_ids)

  def __getitem__(self, idx):
    return self.input_ids[idx], self.target_ids[idx]



In [29]:
def dataLoader(text, batch_size=4, context_length=256, stride=128, drop_last=True, shuffle=True, num_workers=4):

  tokenizer = tiktoken.get_encoding("gpt2")
  dataset = Gpt2Dataset(text, tokenizer, context_length, stride)

  dataloader = DataLoader(
    dataset=dataset,
    batch_size=batch_size,
    drop_last=drop_last,
    shuffle=shuffle,
    num_workers=num_workers
    )

  return dataloader

In [30]:
dataloader = dataLoader(text, batch_size=8, context_length=4, stride=1, drop_last=True, shuffle=True, num_workers=2)

for x, y in dataloader:
  print("Input_tensor", x)
  print(tokenizer.decode(x[0].tolist()))
  print()
  print("Target_tensor", y)
  print(tokenizer.decode(y[0].tolist()))
  break

Input_tensor tensor([[ 1611,   326,   389,  1364],
        [ 8759,  2763,    11,   351],
        [  198,  1870,   465,  8216],
        [  287,   262, 17423,    12],
        [  393, 28537,  2014,   198],
        [10899,   338, 12036,    13],
        [20136,   373,  3957,   588],
        [ 5779,    11,  1282,   981]])
 kind that are left

Target_tensor tensor([[  326,   389,  1364,  2157],
        [ 2763,    11,   351,   326],
        [ 1870,   465,  8216,  1297],
        [  262, 17423,    12,  3823],
        [28537,  2014,   198,   198],
        [  338, 12036,    13,   843],
        [  373,  3957,   588,   262],
        [   11,  1282,   981,   339]])
 that are left behind


In [ ]:
vocab_size = 50257
output_dim = 256

embeddings = torch.nn.Embedding(vocab_size, output_dim)



In [31]:
vocab_size = 50257
output_dim = 256

embeddings = torch.nn.Embedding(vocab_size, output_dim)
pos_encoding_layer = torch.nn.Embedding(context_length, output_dim)
pos_encoding = pos_encoding_layer(torch.arange(context_length))

for input, target in dataloader:
  token_embeddings = embeddings(input)
  input_embedding = token_embeddings + pos_encoding
  break

In [32]:
token_embeddings = embeddings(x)
print(token_embeddings.shape)

torch.Size([8, 4, 256])


**Pos Encoding**


In [35]:
pos_encoding_layer = torch.nn.Embedding(context_length, output_dim)

In [36]:
pos_encoding = pos_encoding_layer(torch.arange(context_length))
print(pos_encoding.shape)

torch.Size([4, 256])


In [37]:
input_embedding = token_embeddings + pos_encoding
print(input_embedding.shape)

torch.Size([8, 4, 256])
